## 简介

本页面为针对特定应用构建 GPT Action 的开发者提供说明和指南。在继续之前，请先熟悉以下信息：
- [GPT Actions 简介](https://platform.openai.com/docs/actions/introduction)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零开始构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

这个 GPT Action 概述了如何连接到 Salesforce，具体来说是 Salesforce Service Cloud。此 Action 中详细定义的 schema 允许用户直接在 ChatGPT 中拉取工单（case）数据并更新工单。为其他 Salesforce Cloud 解决方案创建 Action 的设置流程使用相同的 Connected App 和身份验证设置，但需要不同的 API schema。

### 价值与业务用例示例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力直接连接到 Salesforce

**用例示例**：
- 缩短对客户的平均响应时间
- 缩短排查工单或问题的时间
- 结合 GPT 中的知识和指令，在回复客户时保持更一致的品牌语调

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- [在 Salesforce 中创建 Lightning 应用](https://help.salesforce.com/s/articleView?id=sf.apps_lightning_create.htm&type=5)
- [OAuth 令牌和作用域](https://help.salesforce.com/s/articleView?id=sf.remoteaccess_oauth_tokens_scopes.htm&type=5)
- [Salesforce API 文档](https://developer.salesforce.com/docs/apis)

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 确保你有权限在 Salesforce 中创建应用

## ChatGPT 步骤

### 自定义 GPT 指令

创建自定义 GPT 后，将下方文本复制到 Instructions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

In [ ]:
**Context**: Your purpose is to pull information from Service Cloud, and push updates to cases. A user is going to ask you a question and ask you to make updates.

**Instructions**:
1. When a user asks you to help them solve a case in Service Cloud, ask for the case number and pull the details for the case into the conversation using the getCaseDetailsFromNumber action.
2. If the user asks you to update the case details, use the action updateCaseStatus.

**Example**: 
User: Help me solve case 00001104 in Service Cloud.

### OpenAPI Schema

创建自定义 GPT 后，将下方文本复制到 Actions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

In [ ]:
openapi: 3.1.0
info:
  title: Salesforce Service Cloud Case Update API
  description: API for updating the status of Service Cloud tickets (cases) in Salesforce.
  version: 1.0.3
servers:
  - url: https://your_instance.my.salesforce.com
    description: Base URL for your Salesforce instance (replace 'your_instance' with your actual Salesforce domain)
paths:
  /services/data/v60.0/sobjects/Case/{CaseId}:
    patch:
      operationId: updateCaseStatus
      summary: Updates the status of a Service Cloud case
      description: Updates the status of a Service Cloud ticket based on the case ID number.
      parameters:
        - name: CaseId
          in: path
          required: true
          description: The ID of the case to update.
          schema:
            type: string
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                Status:
                  type: string
                  description: The new status of the case.
      responses:
        '204':
          description: Successfully updated the case status
        '400':
          description: Bad request - invalid input or case ID not found
        '401':
          description: Unauthorized - authentication required
        '404':
          description: Not Found - case ID does not exist
    delete:
      operationId: deleteCase
      summary: Deletes a Service Cloud case
      description: Deletes a Service Cloud ticket based on the case ID number.
      parameters:
        - name: CaseId
          in: path
          required: true
          description: The ID of the case to delete.
          schema:
            type: string
      responses:
        '204':
          description: Successfully deleted the case
        '400':
          description: Bad request - invalid case ID
        '401':
          description: Unauthorized - authentication required
        '404':
          description: Not Found - case ID does not exist
  /services/data/v60.0/query:
    get:
      operationId: getCaseDetailsFromNumber
      summary: Retrieves case details using a case number
      description: Retrieves the details of a Service Cloud case associated with a given case number.
      parameters:
        - name: q
          in: query
          required: true
          description: SOQL query string to find the Case details based on Case Number.
          schema:
            type: string
            example: "SELECT Id, CaseNumber, Status, Subject, Description FROM Case WHERE CaseNumber = '123456'"
      responses:
        '200':
          description: Successfully retrieved the case details
          content:
            application/json:
              schema:
                type: object
                properties:
                  totalSize:
                    type: integer
                  done:
                    type: boolean
                  records:
                    type: array
                    items:
                      type: object
                      properties:
                        Id:
                          type: string
                        CaseNumber:
                          type: string
                        Status:
                          type: string
                        Subject:
                          type: string
                        Description:
                          type: string
        '400':
          description: Bad request - invalid query
        '401':
          description: Unauthorized - authentication required
        '404':
          description: Not Found - case number does not exist


## 身份验证说明

以下是为该第三方应用设置身份验证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，详细了解这一步的工作方式。

### Action 前置步骤

在 ChatGPT 中设置身份验证之前，请先在应用中完成以下步骤。

在 ChatGPT 中设置身份验证之前，请先在应用中完成以下步骤。
1. 进入 Salesforce Setup

![gptactions_salesforce.png](../../../images/gpt_actions_salesforce_setup.png)

2. 搜索“App Manager”


![gptactions_salesforce.png](../../../images//gpt_actions_salesforce_manager.png)

3. 点击“New Connected App”
4. 输入 Connected App 名称
5. 输入联系邮箱（你的邮箱）
6. 勾选启用 OAuth 设置的复选框
7. 填写回调 URL（暂时使用 https://chat.openai.com/aip//oauth/callback 之类的占位符，稍后在 ChatGPT 中创建 Action 时再更新它）


![gptactions_salesforce.png](../../../images//gpt_actions_salesforce_oauth2.png)

8. 选择“Selected OAuth Scopes”并授予适当的权限。请根据你的内部安全策略确定作用域。


![gptactions_salesforce.png](../../../images//gpt_actions_salesforce_scope.png)

9. 确保勾选以下复选框：
- Enable Client Credentials Flow
- Enable Authorization Code and Credentials FLow
- Enable Token Exchange Flow
10. 确保取消勾选以下复选框：
- Require Proof Key for Code Exchange (PKCE) Extension for Supported Authorization Flows


![gptactions_salesforce.png](../../../images/gpt_actions_salesforce_settings_condensed.png) 

11. 保存新的 Connected App
12. 在“Consumer Key and Secret”下点击“Manage Consumer Details”。使用发送到你账户邮箱的验证码验证访问权限，然后复制密钥和密钥值。
- Salesforce Consumer Key = ChatGPT Client ID
- Salesforce Consumer Secret = ChatGPT Client Secret


![gptactions_salesforce.png](../../../images//gpt_actions_salesforce_credentials.png)

13. 返回应用页面
14. 点击“Manage”
15. 点击“Edit Policies”
16. 在 OAuth Policies 下，勾选“Enable Token Exchange Flow”复选框


![gptactions_salesforce.png](../../../images//gpt_actions_salesforce_token.png)

17. 点击保存！

### 在 ChatGPT 中

在 ChatGPT 中，点击“Authentication”并选择 **“OAuth”**。填写以下信息。

- **Client ID**：使用上述步骤中的 Client ID
- **Client Secret**：使用上述步骤中的 Client Secret
- **Authorization URL**：https://[inserturlhere].my.salesforce.com/services/oauth2/authorize
- **Token URL**：https://[inserturlhere].my.salesforce.com/services/oauth2/token
- **Scope**：full
- **Token**：Default (POST)

### Action 后续步骤


在 ChatGPT 中设置好身份验证后，请在应用中按以下步骤完成 Action 的配置。

- 从 GPT Action 中复制回调 URL
- 返回 Salesforce 中你的 Connected App，添加该回调 URL。


### 常见问题与故障排查

- *回调 URL 错误：* 如果你在 ChatGPT 中遇到回调 URL 错误，请仔细查看上面的截图。你需要将回调 URL 直接添加到 Salesforce 中，Action 才能正确完成身份验证
- *内部服务器错误：* 确保在 Connected App 的 OAuth 设置中正确勾选和/或取消勾选了所有复选框。

*你希望我们优先支持哪些集成？我们的集成中是否存在错误？请在我们的 github 中提交 PR 或 issue，我们会进行查看。*
